# Thư viện và dữ liệu

In [1]:
import numpy as np
import pandas as pd
from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics
from prophet.plot import plot_cross_validation_metric
import matplotlib.pyplot as plt

In [1]:
df        = pd.read_csv('../tnbike_data.csv', parse_dates=['ds'])
future_df = pd.read_csv('../future.csv', parse_dates=['ds'])
future_df.head()

In [1]:
df = pd.concat([df, future_df]).reset_index(drop=True)
df.tail()

In [1]:
df = df.rename(columns={'revenue': 'y'})
df.head(1)

In [1]:
df['ds'] = pd.to_datetime(df['ds'])
df.ds

# Ngày lễ

In [1]:
dates_tet = pd.to_datetime(df[df.is_tet == 1].ds)
tet = pd.DataFrame({'holiday': 'tet_nguyen_dan', 'ds': dates_tet, 'lower_window': -7, 'upper_window': 7})

In [1]:
dates_thieu_nhi = pd.to_datetime(df[df.is_thieu_nhi == 1].ds)
thieu_nhi = pd.DataFrame({'holiday': 'ngay_thieu_nhi', 'ds': dates_thieu_nhi, 'lower_window': -14, 'upper_window': 3})

In [1]:
thieu_nhi

In [1]:
holidays = pd.concat([tet, thieu_nhi])
holidays

In [1]:
df = df.drop(['is_tet', 'is_thieu_nhi'], axis=1)

In [1]:
training  = df.iloc[:-3, :]
future_df = df.iloc[-3:, :]

In [1]:
parameters = pd.read_csv('best_params_prophet.csv', index_col=0)
parameters

In [1]:
changepoint_prior_scale = float(parameters.loc['changepoint_prior_scale'][0])
holidays_prior_scale    = float(parameters.loc['holidays_prior_scale'][0])
seasonality_prior_scale = float(parameters.loc['seasonality_prior_scale'][0])
seasonality_mode        = parameters.loc['seasonality_mode'][0]

In [1]:
m = Prophet(holidays=holidays, seasonality_mode=seasonality_mode,
            seasonality_prior_scale=seasonality_prior_scale,
            holidays_prior_scale=holidays_prior_scale,
            changepoint_prior_scale=changepoint_prior_scale)
m.add_seasonality(name='monthly', period=30.5, fourier_order=5)
m.fit(training)

# Tính dừng

In [1]:
from statsmodels.tsa.stattools import adfuller
pvalue = adfuller(x=training['y'].dropna())[1]
if pvalue < 0.05:
    print(f'Chuỗi DỪNG. P-Value = {pvalue:.3f}')
else:
    print(f'Chuỗi KHÔNG DỪNG. P-Value = {pvalue:.3f}')

In [1]:
pvalue = adfuller(training['y'].diff().dropna())[1]
if pvalue < 0.05:
    print(f'DỪNG sau sai phân. P-Value = {pvalue:.3f}')
else:
    print(f'KHÔNG DỪNG sau sai phân. P-Value = {pvalue:.3f}')

# Dự báo

In [1]:
future = m.make_future_dataframe(periods=3, freq='MS')
future.tail()

In [1]:
forecast = m.predict(future)
forecast[['ds','yhat','yhat_lower','yhat_upper']].tail(3)

In [1]:
m.plot_components(forecast);

# Đánh giá

In [1]:
df_cv = cross_validation(m, horizon='90 days', period='30 days', initial='90 days')
performance_metrics(df_cv).head()

In [1]:
print(f'RMSE: {round(performance_metrics(df_cv)["rmse"].mean(), 0):,.0f}')
print(f'MAPE: {100*round(performance_metrics(df_cv)["mape"].mean(), 3):.1f}%')

In [1]:
plot_cross_validation_metric(df_cv, metric='rmse');

# Xuất kết quả

In [1]:
predictions_prophet = forecast[['ds','yhat','yhat_lower','yhat_upper']].tail(3)
predictions_prophet

In [1]:
predictions_prophet.to_csv('Ensemble/predictions_prophet.csv', index=False)
print('✅ Đã lưu predictions_prophet.csv')